# 01 - Data Ingestion

**Objective:** Load the Iris dataset from raw files, clean, encode the target (3 classes), and save processed data.

**Dataset:** UCI Iris — 150 samples, 4 features, 3-class target (Setosa / Versicolor / Virginica)

**Steps:**
1. Load raw data from CSV
2. Encode target as 0/1/2
3. Check data quality (missing values, dtypes)
4. Save processed data


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")


### Dataset Attributes

Understanding each column helps you make better decisions during cleaning and feature engineering.

- **sepal_length:** Sepal length in cm (numeric)
- **sepal_width:** Sepal width in cm (numeric)
- **petal_length:** Petal length in cm (numeric)
- **petal_width:** Petal width in cm (numeric)
- **class:** Iris species — "Iris-setosa", "Iris-versicolor", or "Iris-virginica" (categorical target)


In [ ]:
# TODO: Load the raw CSV data into a DataFrame
# The iris.data file has no header row, so we pass header=None and assign column names ourselves.
# All five values are comma-separated with no missing values in the original.

RAW_DIR = Path("../data/raw")

attribute_names = ["sepal_length", "sepal_width", "petal_length", "petal_width", "class"]

# TODO: Load the data with pd.read_csv()
# Use header=None and names=attribute_names.
# After loading, check the shape and preview the first few rows.
# Also run .info() to confirm all numeric columns are float64.

# df = pd.read_csv(RAW_DIR / "iris.data", header=None, names=attribute_names)
# print(f"Shape: {df.shape}")
# print(df.head())
# df.info()


In [ ]:
# TODO: Encode the target variable as integers
# The class column has three unique values: "Iris-setosa", "Iris-versicolor", "Iris-virginica".
# We map these to 0, 1, 2 so models can process them numerically.
# Label encoding (0/1/2) is fine here because the classes are balanced and there is no
# ordinal relationship — any order works for a 3-class classifier that uses softmax.

# class_map = {"Iris-setosa": 0, "Iris-versicolor": 1, "Iris-virginica": 2}
# df["class"] = df["class"].map(class_map)

# TODO: Check the class distribution
# The dataset has 50 samples per class — verify this stayed balanced after encoding.

# print(df["class"].value_counts().sort_index())


In [ ]:
# TODO: Check for missing values in each column
# Use df.isnull().sum() to get a count of NaN entries per column.
# The original iris.data has no missing values, but it's good practice to verify.

# print(df.isnull().sum())

# TODO: Handle missing values if any are found
# Two common strategies:
#   - df.dropna() — removes any row with a missing value (simple, but you lose data)
#   - df.fillna(value) — replaces missing entries with something like the column mean or median
# Choose the approach that makes sense for your data.

# TODO: Verify the final data is clean
# print(f"Final shape: {df.shape}")
# print(df.head())


In [ ]:
# TODO: Save the processed data to data/processed/clean_data.csv
# Use df.to_csv() with index=False so we don't write the DataFrame index as a column.

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# df.to_csv(PROCESSED_DIR / "clean_data.csv", index=False)
# print("Clean data saved successfully")


In [ ]:
# TODO: Verify the saved file by loading it back into a new DataFrame
# This step confirms the file wasn't corrupted during the write process.
# Compare the shape of the loaded data to the original shape.

# df_check = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
# print(f"Loaded data shape: {df_check.shape}")
# assert df_check.shape == df.shape, "Shape mismatch after save-load cycle"
# print("Verification passed: data round-trips correctly")


## 2. Relational Data — Joining Tables

In real-world projects, data is rarely handed to you in a single tidy CSV. 
It is often spread across multiple tables in a database — a **fact** table 
and several **dimension** tables, linked by key columns.

To build your feature matrix you need to **join** these tables back together.

In this section you will work with a relational version of the Iris data,
split into three CSV files that simulate a botanical database:

| File | Content | Key column |
|------|---------|------------|
| `species.csv` | Species class label | `sample_id` |
| `sepal.csv` | Sepal measurements | `sepal_id` |
| `petal.csv` | Petal measurements | `petal_id` |

Some IDs appear in only some tables — exactly like a real database where 
certain measurements were not recorded or have extra lab entries. 
Your job is to re-assemble the full dataset.

In [ ]:
# Load the three relational tables
RELATIONAL_DIR = Path("../data/raw/relational")

species = pd.read_csv(RELATIONAL_DIR / "species.csv")
sepal = pd.read_csv(RELATIONAL_DIR / "sepal.csv")
petal = pd.read_csv(RELATIONAL_DIR / "petal.csv")

# TODO: Inspect each table
# Check the shape and first few rows.
# How many IDs does each table have? Are the ID ranges the same?

# print(f"Species: {species.shape}")
# print(f"Sepal: {sepal.shape}")
# print(f"Petal: {petal.shape}")
# print(species.head())
# print(sepal.columns.tolist())


In [ ]:
# TODO: Merge species with sepal measurements
# Use pd.merge() with left_on="sample_id" and right_on="sepal_id".
# Use a LEFT JOIN to keep all specimens.
# Count how many rows have NaN for sepal columns.

# merged = pd.merge(species, sepal, left_on="sample_id", right_on="sepal_id", how="left")
# print(f"After merge: {merged.shape}")
# nan_sepal = merged['sepal_length'].isna().sum()
# print(f"Missing sepal: {nan_sepal}")

# TODO: Now try an INNER JOIN. How many rows do you lose? Why?


In [ ]:
# TODO: Merge in the petal measurements
# Chain a second merge to bring in petal data.

# full = pd.merge(merged, petal, left_on="sample_id", right_on="petal_id", how="left")
# print(full.head())

# TODO: Check how many rows are missing petal data
# missing_petal = full['petal_length'].isna().sum()
# print(f"Missing petal: {missing_petal}")


In [ ]:
# TODO: Compare to the original single-table shape
# The original iris.data has 150 rows with 4 features + class.
# After splitting and rejoining, how many complete rows do you have?

# complete = full.dropna()
# print(f"Complete rows after join: {len(complete)}")

# HINT: This tells you how many specimens have all three data sources.
# Is there a specimen with both sepal AND petal missing?


### Exercises

1. **Label encoding vs one-hot**: We used `.map({...})` to encode 3 classes as 0/1/2. Would `pd.get_dummies()` work too? When might one-hot encoding a target variable cause problems? (HINT: Think about how softmax or distance-based models interpret the target.)
2. **Which JOIN?**: You used LEFT JOINs above. Replace them with INNER JOINs. How many rows do you lose overall? Why?
3. **Fake data detective**: The sepal and petal tables contain artificially generated rows (IDs > 150). How many can you find? What heuristic did you use?
4. **Missing value experiment**: Intentionally introduce NaN with `df.loc[0:4, "sepal_length"] = np.nan`. Compare `dropna()` vs `fillna(df["sepal_length"].mean())`. How does each strategy affect the shape and the class distribution?
5. **Feature engineering**: Create `sepal_area = sepal_length * sepal_width` and `petal_area = petal_length * petal_width`. Use `groupby("class")[["sepal_area", "petal_area"]].describe()`. Do these composite features separate the classes better than the raw measurements?
6. **Merge on the wrong key**: Try merging `species` with `petal` on `class` instead of ID. What happens to the row count? Why is this dangerous?
7. **Save the reloaded version**: After merging all three tables back, save the result to `data/processed/clean_data.csv`. Verify the shape matches the original.
8. **What would break?**: If `iris.data` had a header row (column names as the first line of data), which cell would fail first? What error would it produce? How would you fix it?